In [7]:

data <- read.csv("C:/Users/HP/Downloads/FALL25IE423ProjectData_v2/FALL25IE423ProjectData_v2.csv")

# Fix column names created by read.csv
colnames(data) <- gsub("\\.\\.\\.", "_", colnames(data))
head(data)
data.columns <- colnames(data)
print(data.columns)

,match_id,halftime,minute,GOALS_away,GOALS_home,REDCARDS_away,REDCARDS_home,BALL_POSSESSION_away,BALL_POSSESSION_home,PASSES_away,⋯,DANGEROUS_ATTACKS_away,DANGEROUS_ATTACKS_home,SUBSTITUTIONS_away,SUBSTITUTIONS_home,YELLOWCARDS_away,YELLOWCARDS_home,FOULS_away,FOULS_home,INJURIES_away,INJURIES_home
,<int>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,19134475,1st-half,0,0,0,0,0,50,50,0,⋯,0,0,0,0,0,0,0,0,0,0
2,19134475,1st-half,1,0,0,0,0,33,67,5,⋯,0,0,0,0,0,0,2,1,0,0
3,19134475,1st-half,2,0,0,0,0,33,67,5,⋯,0,1,0,0,0,0,2,1,0,0
4,19134475,1st-half,3,0,0,0,0,33,67,5,⋯,0,2,0,0,0,0,2,1,0,0
5,19134475,1st-half,4,0,0,0,0,33,67,5,⋯,0,2,0,0,0,0,2,1,0,0
6,19134475,1st-half,5,0,0,0,0,24,76,15,⋯,0,2,0,0,0,0,2,1,0,0


 [1] "match_id"               "halftime"               "minute"                
 [4] "GOALS_away"             "GOALS_home"             "REDCARDS_away"         
 [7] "REDCARDS_home"          "BALL_POSSESSION_away"   "BALL_POSSESSION_home"  
[10] "PASSES_away"            "PASSES_home"            "SUCCESSFUL_PASSES_away"
[13] "SUCCESSFUL_PASSES_home" "KEY_PASSES_away"        "KEY_PASSES_home"       
[16] "ATTACKS_away"           "ATTACKS_home"           "CORNERS_away"          
[19] "CORNERS_home"           "PENALTIES_away"         "PENALTIES_home"        
[22] "SHOTS_TOTAL_away"       "SHOTS_TOTAL_home"       "SHOTS_ON_TARGET_away"  
[25] "SHOTS_ON_TARGET_home"   "DANGEROUS_ATTACKS_away" "DANGEROUS_ATTACKS_home"
[28] "SUBSTITUTIONS_away"     "SUBSTITUTIONS_home"     "YELLOWCARDS_away"      
[31] "YELLOWCARDS_home"       "FOULS_away"             "FOULS_home"            
[34] "INJURIES_away"          "INJURIES_home"         


In [9]:
############################################
## REQUIRED LIBRARY
############################################
library(zoo)

############################################
## HELPER FUNCTIONS
############################################

half_to_num <- function(x) {
  ifelse(x %in% c("1H", 1, "H1"), 1, 2)
}

cumulative_to_events <- function(x) {
  c(0, pmax(0, diff(x)))
}

############################################
## STEP 1: TRUE LABEL
## red_t_plus_1 = 1 if ANY red occurs next minute
############################################

add_redcard_t_plus_1 <- function(df) {

  df <- df[order(df$match_id,
                 half_to_num(df$halftime),
                 df$minute), ]

  red_any_event <- cumulative_to_events(
    df$REDCARDS_home + df$REDCARDS_away
  )

  df$red_t_plus_1 <- ave(
    red_any_event,
    df$match_id,
    FUN = function(x) c(x[-1], 0)
  )

  # force binary
  df$red_t_plus_1 <- as.integer(df$red_t_plus_1 > 0)

  df
}

############################################
## STEP 2: RELATIVE DISCIPLINE FEATURES
############################################

build_relative_features <- function(df) {

  df <- df[order(df$match_id,
                 half_to_num(df$halftime),
                 df$minute), ]

  # Event-level signals
  df$home_foul_evt   <- cumulative_to_events(df$FOULS_home)
  df$away_foul_evt   <- cumulative_to_events(df$FOULS_away)

  df$home_yellow_evt <- cumulative_to_events(df$YELLOWCARDS_home)
  df$away_yellow_evt <- cumulative_to_events(df$YELLOWCARDS_away)

  # Pressure score (yellow weighted heavier)
  df$home_pressure_raw <- df$home_foul_evt + 2 * df$home_yellow_evt
  df$away_pressure_raw <- df$away_foul_evt + 2 * df$away_yellow_evt

  # Rolling pressure (last 5 minutes)
  df$home_pressure <- ave(
    df$home_pressure_raw,
    df$match_id,
    FUN = function(x) rollsum(x, 4, fill = 0, align = "right")
  )

  df$away_pressure <- ave(
    df$away_pressure_raw,
    df$match_id,
    FUN = function(x) rollsum(x, 4, fill = 0, align = "right")
  )

  # Relative imbalance
  df$pressure_imbalance <- abs(
    df$home_pressure - df$away_pressure
  )

  df
}

############################################
## STEP 3: PREDICT RED AT t+1
## STRICT, LOW-FP LOGIC
############################################

predict_red_from_imbalance <- function(df_match,
                                       imbalance_th = 6,
                                       yellow_th = 1,
                                       persist = 4,
                                       min_minute = 10) {

  n <- nrow(df_match)
  pred <- rep(0L, n)
  streak <- 0L

  total_yellow_evt <- df_match$home_yellow_evt +
                      df_match$away_yellow_evt

  for (i in seq_len(n - 1)) {

    # ignore very early minutes
    if (df_match$minute[i] < min_minute) {
      streak <- 0L
      next
    }

    if (df_match$pressure_imbalance[i] >= imbalance_th ||
        total_yellow_evt[i] >= yellow_th) {
      streak <- streak + 1L
    } else {
      streak <- 0L
    }

    if (streak >= persist) {
      pred[i] <- 1L
    }
  }

  df_match$pred_red_t_plus_1 <- pred
  df_match
}

############################################
## STEP 4: FULL PIPELINE
############################################

run_redcard_pipeline <- function(data,
                                 imbalance_th = 6,
                                 yellow_th = 1,
                                 persist = 4,
                                 min_minute = 10) {

  data <- add_redcard_t_plus_1(data)
  data <- build_relative_features(data)

  out <- do.call(
    rbind,
    lapply(
      split(data, data$match_id),
      predict_red_from_imbalance,
      imbalance_th = imbalance_th,
      yellow_th = yellow_th,
      persist = persist,
      min_minute = min_minute
    )
  )

  out
}

############################################
## STEP 5: RUN MODEL
############################################

result <- run_redcard_pipeline(
  data,
  imbalance_th = 3,
  yellow_th = 1,
  persist = 2,
  min_minute = 10
)

############################################
## STEP 6: EVALUATION
############################################

# Confusion matrix
table(
  Prediction = result$pred_red_t_plus_1,
  Truth      = result$red_t_plus_1
)

# Precision & Recall (%)
TP <- sum(result$pred_red_t_plus_1 == 1 &
          result$red_t_plus_1 == 1)

FP <- sum(result$pred_red_t_plus_1 == 1 &
          result$red_t_plus_1 == 0)

FN <- sum(result$pred_red_t_plus_1 == 0 &
          result$red_t_plus_1 == 1)

precision_pct <- round(100 * TP / (TP + FP), 2)
recall_pct    <- round(100 * TP / (TP + FN), 2)

list(
  TP = TP,
  FP = FP,
  FN = FN,
  precision_percent = precision_pct,
  recall_percent    = recall_pct
)

############################################
## STEP 7: INTERPRETABLE WARNINGS
############################################

# Moments where the model warns a red is coming next minute
result[result$pred_red_t_plus_1 == 1,
       c("match_id",
         "minute",
         "home_pressure",
         "away_pressure",
         "pressure_imbalance",
         "red_t_plus_1")]



Attaching package: 'zoo'


The following objects are masked from 'package:base':

    as.Date, as.Date.numeric




          Truth
Prediction     0     1
         0 54302   120
         1  4775    53

$TP
[1] 53

$FP
[1] 4775

$FN
[1] 120

$precision_percent
[1] 1.1

$recall_percent
[1] 30.64

,match_id,minute,home_pressure,away_pressure,pressure_imbalance,red_t_plus_1
,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>
19095230.1055,19095230,48,4,0,4,0
19095230.1060,19095230,48,4,0,4,0
19095230.1056,19095230,49,4,0,4,0
19095230.1061,19095230,49,4,1,3,0
19095230.1057,19095230,50,4,1,3,0
19095230.1072,19095230,60,5,0,5,0
19095230.1073,19095230,61,4,0,4,0
19095230.1074,19095230,62,4,0,4,0
19095230.1075,19095230,63,1,2,1,0


Looks at how many did we correctly predict. Makes one prediction per red card

In [24]:
############################################
## REQUIRED LIBRARY
############################################
library(zoo)

############################################
## HELPERS
############################################

half_to_num <- function(x) {
  ifelse(x %in% c("1H", 1, "H1"), 1, 2)
}

cumulative_to_events <- function(x) {
  c(0, pmax(0, diff(x)))
}

############################################
## STEP 1: RED CARD EVENTS
############################################

add_red_events <- function(df) {

  df <- df[order(df$match_id,
                 half_to_num(df$halftime),
                 df$minute), ]

  df$red_event <- cumulative_to_events(
    df$REDCARDS_home + df$REDCARDS_away
  )

  df
}

############################################
## STEP 2: PRESSURE FEATURES
############################################

build_pressure_features <- function(df) {

  df <- df[order(df$match_id,
                 half_to_num(df$halftime),
                 df$minute), ]

  df$home_foul_evt   <- cumulative_to_events(df$FOULS_home)
  df$away_foul_evt   <- cumulative_to_events(df$FOULS_away)

  df$home_yellow_evt <- cumulative_to_events(df$YELLOWCARDS_home)
  df$away_yellow_evt <- cumulative_to_events(df$YELLOWCARDS_away)

  df$home_pressure_raw <- df$home_foul_evt + 2.25 * df$home_yellow_evt
  df$away_pressure_raw <- df$away_foul_evt + 2.25 * df$away_yellow_evt

  df$home_pressure <- ave(
    df$home_pressure_raw,
    df$match_id,
    FUN = function(x) rollsum(x, 3, fill = 0, align = "right")
  )

  df$away_pressure <- ave(
    df$away_pressure_raw,
    df$match_id,
    FUN = function(x) rollsum(x, 3, fill = 0, align = "right")
  )

  df$pressure_imbalance <- abs(
    df$home_pressure - df$away_pressure
  )

  df$total_yellow_5 <- ave(
    df$home_yellow_evt + df$away_yellow_evt,
    df$match_id,
    FUN = function(x) rollsum(x, 4, fill = 0, align = "right")
  )

  df
}

############################################
## STEP 3: ONE ALARM PER RED (WITH PERSISTENCE)
############################################

predict_one_alarm_per_red <- function(df_match,
                                      imbalance_th = 4,
                                      yellow_th = 1,
                                      persist = 2,
                                      min_minute = 10) {

  n <- nrow(df_match)
  pred <- rep(0L, n)

  alarm_active <- FALSE
  streak <- 0L

  for (i in seq_len(n)) {

    # Reset after a red card
    if (df_match$red_event[i] == 1) {
      alarm_active <- FALSE
      streak <- 0L
      next
    }

    if (alarm_active) next
    if (df_match$minute[i] < min_minute) {
      streak <- 0L
      next
    }

    condition_met <-
      df_match$pressure_imbalance[i] >= imbalance_th ||
      df_match$total_yellow_5[i] >= yellow_th

    if (condition_met) {
      streak <- streak + 1L
    } else {
      streak <- 0L
    }

    if (streak >= persist) {
      pred[i] <- 1L
      alarm_active <- TRUE   # 🔒 block further alarms
      streak <- 0L
    }
  }

  df_match$pred_red_alarm <- pred
  df_match
}

############################################
## STEP 4: PIPELINE
############################################

run_pipeline <- function(data,
                         imbalance_th = 4,
                         yellow_th = 1,
                         persist = 2,
                         min_minute = 10) {

  data <- add_red_events(data)
  data <- build_pressure_features(data)

  out <- do.call(
    rbind,
    lapply(
      split(data, data$match_id),
      predict_one_alarm_per_red,
      imbalance_th = imbalance_th,
      yellow_th = yellow_th,
      persist = persist,
      min_minute = min_minute
    )
  )

  out
}

############################################
## STEP 5: PREDICTION-LEVEL TIMING
############################################

label_prediction_timing <- function(df) {

  preds <- df[df$pred_red_alarm == 1, ]
  reds  <- df[df$red_event == 1, ]

  timing <- character(nrow(preds))

  for (i in seq_len(nrow(preds))) {

    mid <- preds$match_id[i]
    t_p <- preds$minute[i]

    next_red <- reds$minute[
      reds$match_id == mid & reds$minute > t_p
    ]

    if (length(next_red) == 0) {
      timing[i] <- "no_prediction"
      next
    }

    delta <- min(next_red) - t_p

    timing[i] <- ifelse(delta <= 10, "0-10_minutes_early",
                 ifelse(delta <= 15, "10-15_minutes_early",
                 ifelse(delta <= 20, "15-20_minutes_early",
                 ifelse(delta <= 25, "20-25_minutes_early",
                        ">25_minutes_early"))))
  }

  preds$prediction_window <- timing
  preds
}

############################################
## STEP 6: RUN EVERYTHING
############################################

result <- run_pipeline(
  data,
  imbalance_th = 4,
  yellow_th = 2,
  persist = 4,
  min_minute = 10
)

prediction_level <- label_prediction_timing(result)

############################################
## STEP 7: SUMMARY TABLE
############################################

summary_table <- as.data.frame(
  table(prediction_level$prediction_window)
)

colnames(summary_table) <- c("prediction_window", "count")

print("Prediction-level timing summary:")
print(summary_table)


[1] "Prediction-level timing summary:"
    prediction_window count
1   >25_minutes_early    13
2  0-10_minutes_early     6
3 10-15_minutes_early     5
4 15-20_minutes_early     3
5 20-25_minutes_early     2
6       no_prediction   182


PREDICTION PART

In [25]:
predict <- function(test_data, prev_predictions) {
  # ----------------------------------------
  # Hyperparameters
  # ----------------------------------------
  imbalance_th <- 4
  yellow_th    <- 2
  persist      <- 4
  min_minute   <- 10

  # ----------------------------------------
  # Safety check
  # ----------------------------------------
  if (nrow(test_data) < 2) {
    return(0L)
  }

  # ----------------------------------------
  # Sort correctly (important!)
  # ----------------------------------------
  test_data <- test_data[
    order(
      half_to_num(test_data$halftime),
      test_data$minute
    ),
  ]

  i <- nrow(test_data)  # current timestep

  # ----------------------------------------
  # Detect new red card (RESET state)
  # ----------------------------------------
  red_cum <- test_data$REDCARDS_home + test_data$REDCARDS_away
  red_events <- cumulative_to_events(red_cum)

  current_red_count <- sum(red_events)

  if (current_red_count > MODEL_STATE$last_red_count) {
    MODEL_STATE$alarm_active <- FALSE
    MODEL_STATE$streak <- 0L
    MODEL_STATE$last_red_count <- current_red_count
    return(0L)
  }

  # ----------------------------------------
  # Do not predict if already alarmed
  # ----------------------------------------
  if (MODEL_STATE$alarm_active) {
    return(0L)
  }

  # ----------------------------------------
  # Minimum minute constraint
  # ----------------------------------------
  if (test_data$minute[i] < min_minute) {
    MODEL_STATE$streak <- 0L
    return(0L)
  }

  # ----------------------------------------
  # Event-level features
  # ----------------------------------------
  home_foul_evt   <- cumulative_to_events(test_data$FOULS_home)
  away_foul_evt   <- cumulative_to_events(test_data$FOULS_away)

  home_yellow_evt <- cumulative_to_events(test_data$YELLOWCARDS_home)
  away_yellow_evt <- cumulative_to_events(test_data$YELLOWCARDS_away)

  home_pressure_raw <- home_foul_evt + 2 * home_yellow_evt
  away_pressure_raw <- away_foul_evt + 2 * away_yellow_evt

  # ----------------------------------------
  # Rolling sums (manual, no zoo)
  # ----------------------------------------
  last_k <- function(x, k) {
    sum(tail(x, k), na.rm = TRUE)
  }

  home_pressure <- last_k(home_pressure_raw, 3)
  away_pressure <- last_k(away_pressure_raw, 3)

  pressure_imbalance <- abs(home_pressure - away_pressure)
  total_yellow_5 <- last_k(home_yellow_evt + away_yellow_evt, 3)

  # ----------------------------------------
  # Condition check
  # ----------------------------------------
  condition_met <-
    pressure_imbalance >= imbalance_th ||
    total_yellow_5 >= yellow_th

  if (condition_met) {
    MODEL_STATE$streak <- MODEL_STATE$streak + 1L
  } else {
    MODEL_STATE$streak <- 0L
  }

  # ----------------------------------------
  # Fire alarm (ONCE)
  # ----------------------------------------
  if (MODEL_STATE$streak >= persist) {
    MODEL_STATE$alarm_active <- TRUE
    MODEL_STATE$streak <- 0L
    return(1L)
  }

  return(0L)
}
